In [ ]:
# @title
# ==============================================================================
# FLASHGO - ULTIMATE ENTERPRISE LEVEL v3.2.1 (EXTENDED POST-RIDE VERSION)
# TÍCH HỢP: 5D FUZZY LOGIC AI + MARKET PRESSURE + REVERSE GEOCODING + SPLASH SCREEN
# NEW UPDATE: LIVE CHAT, VOIP CALLING & OPTIONAL 5-STAR RATING SYSTEM
# ==============================================================================

!pip install -q scikit-fuzzy flask flask-cors

import threading
import time
import urllib.request
import json
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from flask import Flask, request, jsonify, render_template_string
from werkzeug.serving import make_server
import logging
import random
import math
from datetime import datetime

# Tắt log của server để giao diện sạch sẽ
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

# ==============================================================================
# PHẦN 1: HỆ THỐNG AI LOGIC MỜ 5 CHIỀU (5D FUZZY LOGIC ENGINE)
# -> GIỮ NGUYÊN 100%
# ==============================================================================
thoi_tiet = ctrl.Antecedent(np.arange(0, 11, 0.1), 'thoi_tiet')
giao_thong = ctrl.Antecedent(np.arange(0, 101, 1), 'giao_thong')
thoi_gian = ctrl.Antecedent(np.arange(0, 24.1, 0.1), 'thoi_gian')
ngay_trong_tuan = ctrl.Antecedent(np.arange(1, 8, 1), 'ngay_trong_tuan')
nhu_cau = ctrl.Antecedent(np.arange(0, 101, 1), 'nhu_cau')
he_so_gia = ctrl.Consequent(np.arange(0.5, 5.1, 0.1), 'he_so_gia')

thoi_tiet['rat_dep'] = fuzz.trapmf(thoi_tiet.universe, [0, 0, 1, 2])
thoi_tiet['dep'] = fuzz.trimf(thoi_tiet.universe, [1.5, 3, 4.5])
thoi_tiet['binh_thuong'] = fuzz.trimf(thoi_tiet.universe, [4, 5.5, 7])
thoi_tiet['xau'] = fuzz.trimf(thoi_tiet.universe, [6.5, 8, 9])
thoi_tiet['rat_xau'] = fuzz.trapmf(thoi_tiet.universe, [8.5, 9.5, 10, 10])

giao_thong['rat_vang'] = fuzz.trapmf(giao_thong.universe, [0, 0, 10, 20])
giao_thong['vang'] = fuzz.trimf(giao_thong.universe, [15, 25, 40])
giao_thong['trung_binh'] = fuzz.trimf(giao_thong.universe, [35, 50, 65])
giao_thong['dong'] = fuzz.trimf(giao_thong.universe, [60, 75, 85])
giao_thong['ket_xe'] = fuzz.trapmf(giao_thong.universe, [80, 90, 100, 100])

thoi_gian['dem_khuya'] = fuzz.trapmf(thoi_gian.universe, [0, 0, 4, 6])
thoi_gian['sang_som'] = fuzz.trimf(thoi_gian.universe, [5, 6.5, 8])
thoi_gian['cao_diem_sang'] = fuzz.trimf(thoi_gian.universe, [7, 8.5, 10])
thoi_gian['trua_chieu'] = fuzz.trapmf(thoi_gian.universe, [9, 11, 15, 17])
thoi_gian['cao_diem_chieu'] = fuzz.trimf(thoi_gian.universe, [16, 18, 20])
thoi_gian['buoi_toi'] = fuzz.trapmf(thoi_gian.universe, [19, 21, 24, 24])

ngay_trong_tuan['ngay_thuong'] = fuzz.trapmf(ngay_trong_tuan.universe, [1, 1, 4, 5.5])
ngay_trong_tuan['cuoi_tuan'] = fuzz.trapmf(ngay_trong_tuan.universe, [5, 6, 7, 7])

nhu_cau['thap'] = fuzz.trapmf(nhu_cau.universe, [0, 0, 20, 40])
nhu_cau['trung_binh'] = fuzz.trimf(nhu_cau.universe, [30, 50, 70])
nhu_cau['cao'] = fuzz.trimf(nhu_cau.universe, [60, 75, 85])
nhu_cau['qua_tai'] = fuzz.trapmf(nhu_cau.universe, [80, 90, 100, 100])

he_so_gia['cuc_re'] = fuzz.trapmf(he_so_gia.universe, [0.5, 0.5, 0.7, 0.9])
he_so_gia['re'] = fuzz.trimf(he_so_gia.universe, [0.8, 1.0, 1.2])
he_so_gia['binh_thuong'] = fuzz.trimf(he_so_gia.universe, [1.1, 1.3, 1.5])
he_so_gia['hoi_cao'] = fuzz.trimf(he_so_gia.universe, [1.4, 1.7, 2.0])
he_so_gia['cao'] = fuzz.trimf(he_so_gia.universe, [1.9, 2.3, 2.7])
he_so_gia['rat_cao'] = fuzz.trimf(he_so_gia.universe, [2.5, 3.0, 3.5])
he_so_gia['cuc_cao'] = fuzz.trapmf(he_so_gia.universe, [3.2, 3.7, 4.2, 4.5])
he_so_gia['khung_khieng'] = fuzz.trapmf(he_so_gia.universe, [4.0, 4.5, 5.0, 5.0])

rules = []
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['dem_khuya'] & nhu_cau['thap'], he_so_gia['binh_thuong']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['dem_khuya'] & nhu_cau['cao'], he_so_gia['cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['dem_khuya'] & thoi_tiet['rat_xau'], he_so_gia['rat_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['cao_diem_sang'] & giao_thong['ket_xe'] & nhu_cau['qua_tai'] & thoi_tiet['rat_xau'], he_so_gia['khung_khieng']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['cao_diem_sang'] & giao_thong['ket_xe'] & nhu_cau['cao'] & thoi_tiet['xau'], he_so_gia['cuc_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['cao_diem_sang'] & giao_thong['dong'] & nhu_cau['cao'] & thoi_tiet['binh_thuong'], he_so_gia['rat_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['cao_diem_sang'] & nhu_cau['trung_binh'] & thoi_tiet['dep'], he_so_gia['hoi_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['trua_chieu'] & giao_thong['vang'] & nhu_cau['thap'], he_so_gia['re']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['trua_chieu'] & nhu_cau['cao'] & thoi_tiet['rat_xau'], he_so_gia['cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['cao_diem_chieu'] & giao_thong['ket_xe'] & nhu_cau['qua_tai'] & thoi_tiet['rat_xau'], he_so_gia['khung_khieng']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['cao_diem_chieu'] & giao_thong['dong'] & nhu_cau['cao'] & thoi_tiet['xau'], he_so_gia['cuc_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['ngay_thuong'] & thoi_gian['cao_diem_chieu'] & giao_thong['dong'] & nhu_cau['trung_binh'] & thoi_tiet['dep'], he_so_gia['cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['cuoi_tuan'] & thoi_gian['sang_som'] & nhu_cau['cao'], he_so_gia['hoi_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['cuoi_tuan'] & thoi_gian['cao_diem_sang'] & giao_thong['dong'] & nhu_cau['cao'], he_so_gia['cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['cuoi_tuan'] & thoi_gian['buoi_toi'] & giao_thong['ket_xe'] & nhu_cau['qua_tai'] & thoi_tiet['rat_xau'], he_so_gia['khung_khieng']))
rules.append(ctrl.Rule(ngay_trong_tuan['cuoi_tuan'] & thoi_gian['buoi_toi'] & giao_thong['dong'] & nhu_cau['qua_tai'] & thoi_tiet['dep'], he_so_gia['cuc_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['cuoi_tuan'] & thoi_gian['buoi_toi'] & giao_thong['trung_binh'] & nhu_cau['cao'] & thoi_tiet['binh_thuong'], he_so_gia['rat_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['cuoi_tuan'] & thoi_gian['dem_khuya'] & nhu_cau['cao'], he_so_gia['rat_cao']))
rules.append(ctrl.Rule(ngay_trong_tuan['cuoi_tuan'] & thoi_gian['dem_khuya'] & nhu_cau['qua_tai'] & thoi_tiet['xau'], he_so_gia['cuc_cao']))
rules.append(ctrl.Rule(thoi_tiet['rat_xau'] & giao_thong['ket_xe'] & nhu_cau['qua_tai'], he_so_gia['khung_khieng']))
rules.append(ctrl.Rule(thoi_tiet['xau'] & giao_thong['ket_xe'] & nhu_cau['cao'], he_so_gia['cuc_cao']))
rules.append(ctrl.Rule(thoi_tiet['rat_dep'] & giao_thong['rat_vang'] & nhu_cau['thap'], he_so_gia['cuc_re']))
rules.append(ctrl.Rule(thoi_tiet['dep'] & giao_thong['vang'] & nhu_cau['thap'], he_so_gia['re']))
rules.append(ctrl.Rule(nhu_cau['qua_tai'] & thoi_tiet['dep'], he_so_gia['rat_cao']))
rules.append(ctrl.Rule(nhu_cau['qua_tai'] & thoi_tiet['binh_thuong'], he_so_gia['cuc_cao']))
rules.append(ctrl.Rule(giao_thong['ket_xe'] & nhu_cau['trung_binh'], he_so_gia['cao']))
rules.append(ctrl.Rule(giao_thong['ket_xe'] & nhu_cau['thap'], he_so_gia['hoi_cao']))

pricing_ctrl = ctrl.ControlSystem(rules)
pricing_sim = ctrl.ControlSystemSimulation(pricing_ctrl)

def calculate_5d_fuzzy(weather_val, traffic_val, hour_val, day_val, demand_val):
    try:
        pricing_sim.input['thoi_tiet'] = float(weather_val)
        pricing_sim.input['giao_thong'] = float(traffic_val)
        pricing_sim.input['thoi_gian'] = float(hour_val)
        pricing_sim.input['ngay_trong_tuan'] = float(day_val)
        pricing_sim.input['nhu_cau'] = float(demand_val)
        pricing_sim.compute()
        return round(pricing_sim.output['he_so_gia'], 2)
    except Exception as e:
        print(f"Fuzzy 5D Compute Error: {e}")
        return 1.3

# ==============================================================================
# PHẦN 2: CƠ SỞ DỮ LIỆU ENTERPRISE (MOCKUP DATABASE MỞ RỘNG)
# ==============================================================================
VOUCHERS_DB = {
    "SUPERAPP": {"type": "percent", "value": 0.2, "max_discount": 50000, "desc": "Giảm 20% (Tối đa 50k)"},
    "FREESHIP": {"type": "fixed", "value": 30000, "max_discount": 30000, "desc": "Giảm thẳng 30k"},
    "QUAN1": {"type": "percent", "value": 0.15, "max_discount": 40000, "desc": "Giảm 15% vi vu Trung tâm TPHCM"},
    "VIPMEMBER": {"type": "percent", "value": 0.5, "max_discount": 100000, "desc": "Giảm 50% cho hạng VIP (Tối đa 100k)"},
    "CHAOHE": {"type": "fixed", "value": 15000, "max_discount": 15000, "desc": "Giảm 15k cho cuốc xe mùa hè"},
    "DEMKHUYA": {"type": "percent", "value": 0.25, "max_discount": 60000, "desc": "Giảm 25% cho chuyến xe đêm"},
    "CUOITUAN": {"type": "fixed", "value": 20000, "max_discount": 20000, "desc": "Giảm 20k vi vu cuối tuần"},
    "MUAXUAN": {"type": "percent", "value": 0.1, "max_discount": 25000, "desc": "Giảm 10% lộc xuân"},
    "TANTRUONG": {"type": "fixed", "value": 10000, "max_discount": 10000, "desc": "Giảm 10k cho Học sinh Sinh viên"},
    "SANBAY": {"type": "fixed", "value": 50000, "max_discount": 50000, "desc": "Giảm 50k cuốc đi Sân bay Tân Sơn Nhất"}
}

first_names = ["Nguyễn", "Trần", "Lê", "Phạm", "Hoàng", "Huỳnh", "Phan", "Vũ", "Võ", "Đặng", "Bùi", "Đỗ"]
middle_names = ["Văn", "Thị", "Hoàng", "Minh", "Quốc", "Thanh", "Ngọc", "Tuấn", "Thái", "Hữu"]
last_names = ["A", "B", "C", "D", "Anh", "Tuấn", "Linh", "Trang", "Khoa", "Đạt", "Phát", "Thảo", "Vy"]
car_models = ["Toyota Vios", "Hyundai Accent", "Honda City", "Kia Morning", "Mazda 3", "Mitsubishi Xpander", "VinFast VF e34", "Honda Civic", "Kia Cerato", "Toyota Innova", "Ford Ranger"]

DRIVERS_DB = []
for i in range(1, 41):
    fname = random.choice(first_names)
    mname = random.choice(middle_names)
    lname = random.choice(last_names)
    car = random.choice(car_models)
    plate_num = f"{random.randint(100,999)}.{random.randint(10,99)}"
    plate_prefix = random.choice(["51H", "51G", "51F", "51K", "51C"])

    DRIVERS_DB.append({
        "id": f"D{i:03d}",
        "name": f"{fname} {mname} {lname}",
        "plate": f"{plate_prefix}-{plate_num}",
        "rating": round(random.uniform(4.5, 5.0), 2),
        "trips": random.randint(100, 5000),
        "car": car
    })

# ==============================================================================
# PHẦN 3: FRONT-END UI/UX (HTML, CSS, JS) - CẬP NHẬT TÍNH NĂNG HẬU MÃI (TÙY CHỌN ĐÁNH GIÁ)
# ==============================================================================
app = Flask(__name__)

HTML_CODE = """
<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>FlashGo</title>

    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/css/bootstrap.min.css" rel="stylesheet">
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <link rel="stylesheet" href="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.css" />
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Be+Vietnam+Pro:wght@300;400;500;600;700;800&display=swap" rel="stylesheet">

    <style>
        :root {
            --main-bg: #f1f5f9; --panel-bg: #ffffff; --primary: #0f172a; --primary-light: #1e293b;
            --accent: #3b82f6; --accent-hover: #2563eb; --success: #10b981; --danger: #ef4444;
            --warning: #f59e0b; --text-main: #0f172a; --text-muted: #64748b;
        }
        body { background-color: var(--main-bg); font-family: 'Be Vietnam Pro', sans-serif; overflow-x: hidden; margin: 0; color: var(--text-main); }
        ::-webkit-scrollbar { width: 8px; } ::-webkit-scrollbar-track { background: transparent; } ::-webkit-scrollbar-thumb { background: #cbd5e1; border-radius: 10px; }

        /* --- SPLASH SCREEN STYLES --- */
        #splashScreen { position: fixed; top: 0; left: 0; width: 100vw; height: 100vh; background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); z-index: 99999; display: flex; flex-direction: column; justify-content: center; align-items: center; color: white; transition: opacity 0.8s ease-out, visibility 0.8s; }
        .splash-icon { font-size: 5rem; color: var(--accent); margin-bottom: 20px; animation: float 2s ease-in-out infinite; }
        .splash-title { font-size: 3.5rem; font-weight: 800; letter-spacing: 3px; margin-bottom: 30px; text-shadow: 0 4px 10px rgba(0,0,0,0.5); text-align: center; line-height: 1.2; }
        .splash-badge { display: block; font-size: 1.2rem; color: var(--warning); margin-top: 5px; font-weight: 700; letter-spacing: 5px; }
        .spinner-ring { width: 60px; height: 60px; border: 5px solid rgba(255,255,255,0.1); border-top-color: var(--accent); border-radius: 50%; animation: spin 1s cubic-bezier(0.55, 0.15, 0.45, 0.85) infinite; margin-bottom: 20px; }
        .splash-status { font-size: 1.1rem; color: #94a3b8; font-weight: 500; min-height: 25px; transition: opacity 0.3s; }

        @keyframes spin { 100% { transform: rotate(360deg); } }
        @keyframes float { 0%, 100% { transform: translateY(0); } 50% { transform: translateY(-15px); } }

        /* --- UI STYLES --- */
        .app-header { background: var(--primary); color: white; padding: 15px 30px; display: flex; justify-content: space-between; align-items: center; box-shadow: 0 4px 20px rgba(0,0,0,0.1); position: relative; z-index: 10; }
        .app-logo { font-size: 1.5rem; font-weight: 800; letter-spacing: 1px; display: flex; align-items: center; gap: 10px; }
        .app-logo i { color: var(--accent); }
        .user-profile { display: flex; align-items: center; gap: 10px; font-weight: 500;}
        .user-avatar { width: 40px; height: 40px; border-radius: 50%; background: var(--accent); display: flex; justify-content: center; align-items: center; font-weight: bold;}

        .main-container { padding: 20px; max-width: 1600px; margin: auto; }

        .dashboard-panel { background: white; border-radius: 16px; padding: 20px; box-shadow: 0 5px 20px rgba(0,0,0,0.05); margin-bottom: 20px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 15px; border-left: 5px solid var(--accent); }
        .metric-box { display: flex; align-items: center; gap: 15px; padding: 10px 20px; background: #f8fafc; border-radius: 12px; min-width: 200px; }
        .metric-icon { font-size: 2rem; color: var(--accent); }
        .metric-data h6 { margin: 0; font-size: 0.85rem; color: var(--text-muted); text-transform: uppercase; font-weight: 700; letter-spacing: 0.5px;}
        .metric-data h3 { margin: 0; font-size: 1.4rem; font-weight: 800; color: var(--primary);}

        #map { height: 75vh; width: 100%; border-radius: 16px; box-shadow: 0 10px 25px rgba(0,0,0,0.08); z-index: 1; border: 1px solid #e2e8f0; }
        .leaflet-routing-container { display: none !important; }
        .control-panel { background: var(--panel-bg); height: 75vh; border-radius: 16px; padding: 25px; box-shadow: 0 10px 25px rgba(0,0,0,0.08); overflow-y: auto; border: 1px solid #e2e8f0;}
        .section-title { font-size: 0.95rem; font-weight: 700; color: var(--text-muted); text-transform: uppercase; margin-bottom: 15px; letter-spacing: 0.5px; border-bottom: 2px solid #f1f5f9; padding-bottom: 8px;}

        .search-wrapper { position: relative; margin-bottom: 20px; }
        .search-input { padding-left: 45px; border-radius: 10px; border: 2px solid #e2e8f0; height: 50px; font-weight: 500; transition: all 0.3s; background: #f8fafc;}
        .search-input:focus { box-shadow: 0 0 0 3px rgba(59,130,246,0.15); border-color: var(--accent); background: white;}
        .search-icon { position: absolute; left: 15px; top: 15px; color: #94a3b8; font-size: 1.1rem; z-index: 10;}
        .gps-btn { position: absolute; right: 8px; top: 8px; border-radius: 8px; padding: 5px 12px; font-size: 0.85rem; z-index: 10; font-weight: 600;}

        .payment-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-bottom: 20px;}
        .payment-card { border: 2px solid #e2e8f0; border-radius: 10px; padding: 12px; text-align: center; cursor: pointer; transition: all 0.2s; font-weight: 600; font-size: 0.9rem; display: flex; flex-direction: column; align-items: center; gap: 5px; background: white;}
        .payment-card.active { border-color: var(--accent); background-color: #eff6ff; color: var(--accent); }
        .payment-card i { font-size: 1.5rem; }
        .payment-card[data-val="cash"] i { color: #10b981; } .payment-card[data-val="momo"] i { color: #a855f7; }
        .payment-card[data-val="zalopay"] i { color: #0ea5e9; } .payment-card[data-val="card"] i { color: #f59e0b; }

        .voucher-input-group { display: flex; gap: 10px; margin-bottom: 15px;}
        .voucher-input-group input { text-transform: uppercase; font-weight: bold; border-radius: 8px; border: 2px solid #e2e8f0;}
        .voucher-input-group button { border-radius: 8px; font-weight: bold; padding: 0 20px; background: var(--primary); color: white; border: none;}
        #voucherDesc { font-size: 0.9rem; margin-bottom: 15px; font-weight: 600;}

        .invoice-box { background: linear-gradient(145deg, #0f172a, #1e293b); color: white; border-radius: 16px; padding: 25px; position: relative; overflow: hidden; box-shadow: 0 15px 30px rgba(0,0,0,0.15);}
        .inv-row { display: flex; justify-content: space-between; margin-bottom: 12px; font-size: 0.95rem; color: #cbd5e1; align-items: center;}
        .inv-val { font-weight: 600; color: #fff; font-size: 1.05rem;}
        .inv-discount-row { color: #10b981; font-weight: 600; background: rgba(16, 185, 129, 0.1); padding: 8px 12px; border-radius: 8px; margin-bottom: 12px; display: none; justify-content: space-between; border: 1px dashed rgba(16, 185, 129, 0.3);}
        .inv-total-row { display: flex; justify-content: space-between; margin-top: 15px; padding-top: 15px; border-top: 1px dashed #475569; font-size: 1.6rem; font-weight: 800; color: var(--warning); }
        .btn-booking { background: var(--accent); color: white; border: none; font-weight: 800; padding: 15px; border-radius: 12px; font-size: 1.2rem; transition: background 0.3s; }
        .btn-booking:hover { background: var(--accent-hover); color: white;}

        #toastContainer { position: fixed; top: 20px; right: 20px; z-index: 100000; display: flex; flex-direction: column; gap: 10px; }
        .custom-toast { min-width: 250px; padding: 15px 20px; border-radius: 10px; color: white; font-weight: 600; box-shadow: 0 10px 20px rgba(0,0,0,0.15); display: flex; align-items: center; gap: 12px; transform: translateX(120%); transition: transform 0.4s; }
        .custom-toast.show { transform: translateX(0); }
        .toast-success { background: linear-gradient(135deg, #10b981, #059669); } .toast-error { background: linear-gradient(135deg, #ef4444, #dc2626); }

        .custom-marker { background: none; border: none; }
        .marker-pin { width: 30px; height: 30px; border-radius: 50% 50% 50% 0; background: #c30b82; position: absolute; transform: rotate(-45deg); left: 50%; top: 50%; margin: -15px 0 0 -15px; box-shadow: 0 3px 5px rgba(0,0,0,0.3); }
        .marker-pin::after { content: ''; width: 14px; height: 14px; margin: 8px 0 0 8px; background: #fff; position: absolute; border-radius: 50%; }
        .origin-marker .marker-pin { background: var(--success); } .dest-marker .marker-pin { background: var(--danger); }

        #driverModal { position: fixed; top:0; left:0; width:100vw; height:100vh; background: rgba(15, 23, 42, 0.9); z-index: 10000; display: none; justify-content: center; align-items: center; backdrop-filter: blur(5px);}
        .driver-card { background: white; width: 90%; max-width: 400px; border-radius: 20px; padding: 30px; text-align: center; position: relative; overflow: hidden; box-shadow: 0 25px 50px rgba(0,0,0,0.25);}
        .radar-box { position: relative; width: 120px; height: 120px; margin: 0 auto 20px; border-radius: 50%; background: #eff6ff; border: 2px solid #bfdbfe; overflow: hidden;}
        .radar-sweep { position: absolute; top: 0; left: 50%; width: 50%; height: 50%; background: linear-gradient(90deg, rgba(37,99,235,0.5) 0%, transparent 100%); transform-origin: bottom left; animation: radar 2s linear infinite;}
        .radar-box i { position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%); font-size: 2rem; color: var(--accent); z-index: 2;}
        @keyframes radar { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }

        #activeRidePanel { background: white; border: 2px solid var(--accent); border-radius: 16px; padding: 25px; box-shadow: 0 10px 25px rgba(37,99,235,0.15);}
        .autocomplete-dropdown { position: absolute; top: 100%; left: 0; right: 0; background: white; border: 1px solid #e2e8f0; border-radius: 10px; max-height: 250px; overflow-y: auto; z-index: 1000; box-shadow: 0 10px 20px rgba(0,0,0,0.1); display: none; }
        .autocomplete-item { padding: 12px 15px; cursor: pointer; border-bottom: 1px solid #f1f5f9; display: flex; align-items: flex-start; gap: 10px; transition: background 0.2s;}
        .autocomplete-item:hover { background-color: #f8fafc; }

        /* ====================================================================== */
        /* --- CSS MỚI THÊM CHO CÁC TÍNH NĂNG LIVE CHAT, GỌI ĐIỆN VÀ RATING --- */
        /* ====================================================================== */

        .action-btn-group { display: flex; gap: 10px; margin-bottom: 20px; }
        .action-btn { flex: 1; padding: 12px; border-radius: 12px; font-weight: bold; border: 2px solid; background: white; transition: all 0.3s; cursor: pointer; display: flex; justify-content: center; align-items: center; gap: 8px; font-size: 0.95rem; }
        .btn-call { border-color: var(--success); color: var(--success); }
        .btn-call:hover { background: var(--success); color: white; box-shadow: 0 5px 15px rgba(16, 185, 129, 0.3); transform: translateY(-2px); }
        .btn-chat { border-color: var(--accent); color: var(--accent); }
        .btn-chat:hover { background: var(--accent); color: white; box-shadow: 0 5px 15px rgba(59, 130, 246, 0.3); transform: translateY(-2px); }

        /* Khung Modals Cơ Bản */
        .overlay-modal { position: fixed; top: 0; left: 0; width: 100vw; height: 100vh; background: rgba(15, 23, 42, 0.85); z-index: 100000; display: none; justify-content: center; align-items: center; backdrop-filter: blur(8px); }
        .modal-content-box { background: white; width: 90%; max-width: 420px; border-radius: 24px; overflow: hidden; box-shadow: 0 25px 50px rgba(0,0,0,0.3); display: flex; flex-direction: column; position: relative; animation: slideUp 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275); }
        @keyframes slideUp { from { opacity: 0; transform: translateY(50px); } to { opacity: 1; transform: translateY(0); } }

        /* Giao Diện Nhắn Tin (Live Chat UI) */
        .chat-header { background: var(--primary); color: white; padding: 20px; display: flex; align-items: center; justify-content: space-between; }
        .chat-body { height: 350px; background: #f8fafc; padding: 20px; overflow-y: auto; display: flex; flex-direction: column; gap: 15px; scroll-behavior: smooth;}
        .msg-bubble { max-width: 80%; padding: 12px 16px; border-radius: 18px; font-size: 0.95rem; line-height: 1.4; animation: fadeIn 0.3s; box-shadow: 0 2px 5px rgba(0,0,0,0.05);}
        @keyframes fadeIn { from { opacity: 0; transform: translateY(10px); } to { opacity: 1; transform: translateY(0); } }
        .msg-driver { background: #e2e8f0; color: var(--text-main); align-self: flex-start; border-bottom-left-radius: 4px; }
        .msg-user { background: var(--accent); color: white; align-self: flex-end; border-bottom-right-radius: 4px; }
        .chat-footer { padding: 15px; background: white; border-top: 1px solid #e2e8f0; display: flex; gap: 10px; align-items: center;}
        .chat-input { flex: 1; border: 2px solid transparent; background: #f1f5f9; padding: 12px 20px; border-radius: 20px; outline: none; transition: 0.3s; font-family: inherit;}
        .chat-input:focus { border-color: var(--accent); background: white;}
        .chat-send { background: var(--accent); color: white; border: none; width: 45px; height: 45px; border-radius: 50%; display: flex; justify-content: center; align-items: center; cursor: pointer; transition: 0.2s; box-shadow: 0 4px 10px rgba(59,130,246,0.3);}
        .chat-send:hover { transform: scale(1.08); background: var(--accent-hover); }

        /* Giao Diện Gọi Điện (VoIP Call UI) */
        .call-box { background: linear-gradient(135deg, #1e293b, #0f172a); padding: 40px 20px; text-align: center; color: white; display: flex; flex-direction: column; align-items: center; gap: 20px; border-radius: 30px; box-shadow: 0 25px 50px rgba(0,0,0,0.5); width: 100%; max-width: 350px;}
        .call-avatar { width: 130px; height: 130px; border-radius: 50%; border: 4px solid rgba(255,255,255,0.2); background: #cbd5e1; position: relative; z-index: 2; object-fit: cover;}
        .call-avatar-wrapper { position: relative; margin: 20px 0; }
        .call-avatar-wrapper.ringing::before, .call-avatar-wrapper.ringing::after { content: ''; position: absolute; top: 0; left: 0; right: 0; bottom: 0; border: 2px solid var(--success); border-radius: 50%; z-index: 1; animation: pulseRinging 2s linear infinite; }
        .call-avatar-wrapper.ringing::after { animation-delay: 1s; }
        @keyframes pulseRinging { 0% { transform: scale(1); opacity: 0.8; } 100% { transform: scale(1.6); opacity: 0; } }
        .call-name { font-size: 1.8rem; font-weight: 800; margin: 0; letter-spacing: 0.5px;}
        .call-status-text { font-size: 1.1rem; color: #94a3b8; font-weight: 500;}
        .call-btn-hangup { background: var(--danger); width: 75px; height: 75px; border-radius: 50%; color: white; font-size: 2rem; border: none; display: flex; justify-content: center; align-items: center; cursor: pointer; transition: 0.3s; margin-top: 20px; box-shadow: 0 10px 25px rgba(239, 68, 68, 0.4); }
        .call-btn-hangup:hover { background: #dc2626; transform: translateY(-5px); box-shadow: 0 15px 30px rgba(239, 68, 68, 0.5);}

        /* Giao Diện Đánh Giá Sao (5-Star Rating UI) */
        .rating-box { padding: 40px 30px; text-align: center; }
        .rating-title { font-size: 1.6rem; font-weight: 800; color: var(--primary); margin-bottom: 10px; }
        .rating-subtitle { color: var(--text-muted); margin-bottom: 30px; font-size: 0.95rem; line-height: 1.5;}
        .stars { display: flex; justify-content: center; gap: 15px; flex-direction: row-reverse; margin-bottom: 30px; }
        .star-icon { font-size: 2.5rem; color: #cbd5e1; cursor: pointer; transition: all 0.2s cubic-bezier(0.175, 0.885, 0.32, 1.275); }
        .star-icon:hover, .star-icon:hover ~ .star-icon { color: var(--warning); transform: scale(1.2); text-shadow: 0 0 15px rgba(245, 158, 11, 0.4);}
        .star-icon.active { color: var(--warning); }
        .rating-comment { width: 100%; border: 2px solid #e2e8f0; border-radius: 12px; padding: 15px; outline: none; resize: none; margin-bottom: 25px; font-family: inherit; transition: 0.3s; background: #f8fafc;}
        .rating-comment:focus { border-color: var(--warning); background: white; box-shadow: 0 0 0 4px rgba(245, 158, 11, 0.1);}
        .btn-submit-rating { background: var(--primary); color: white; border: none; padding: 16px; border-radius: 12px; width: 100%; font-weight: 800; font-size: 1.1rem; cursor: pointer; transition: 0.3s; box-shadow: 0 10px 20px rgba(15, 23, 42, 0.2);}
        .btn-submit-rating:hover { background: var(--primary-light); transform: translateY(-2px);}

        @media (max-width: 991px) {
            .control-panel { height: auto; min-height: 50vh; margin-top: 20px;}
            #map { height: 50vh; }
            .dashboard-panel { flex-direction: column; align-items: stretch; }
        }
    </style>
</head>
<body>

    <div id="splashScreen">
        <i class="fa-solid fa-car-side splash-icon"></i>
        <div class="splash-title">
            FlashGo
        </div>
        <div class="spinner-ring"></div>
        <div class="splash-status" id="splashStatus">Đang khởi tạo máy học (AI Engine)...</div>
    </div>

    <div id="toastContainer"></div>

    <header class="app-header">
        <div class="app-logo">
            <i class="fa-solid fa-car-side"></i> FlashGo <span style="font-size: 0.8rem; background: var(--accent); padding: 3px 8px; border-radius: 5px; margin-left: 10px;">ENTERPRISE</span>
        </div>
        <div class="user-profile">
            <span>Hi, Khách Hàng</span>
            <div class="user-avatar">KH</div>
        </div>
    </header>

    <div class="main-container">
        <div class="dashboard-panel">
            <div class="metric-box">
                <i class="fa-regular fa-calendar-days metric-icon text-primary"></i>
                <div class="metric-data">
                    <h6>Ngày hiện tại</h6>
                    <h3 id="dashDay">Thứ 2</h3>
                </div>
            </div>
            <div class="metric-box">
                <i class="fa-solid fa-clock metric-icon text-warning"></i>
                <div class="metric-data">
                    <h6>Giờ hệ thống</h6>
                    <h3 id="dashTime">00:00</h3>
                </div>
            </div>
            <div class="metric-box">
                <i class="fa-solid fa-cloud-sun metric-icon text-info"></i>
                <div class="metric-data">
                    <h6>Thời tiết</h6>
                    <h3 id="dashWeather">Đang quét...</h3>
                </div>
            </div>
            <div class="metric-box">
                <i class="fa-solid fa-chart-line metric-icon text-danger"></i>
                <div class="metric-data">
                    <h6>Nhu cầu (Demand)</h6>
                    <h3 id="dashDemand">Bình thường</h3>
                </div>
            </div>
        </div>

        <div class="row g-4">
            <div class="col-xl-8 col-lg-7">
                <div id="map"></div>
            </div>

            <div class="col-xl-4 col-lg-5">
                <div class="control-panel">
                    <div class="section-title"><i class="fa-solid fa-route"></i> Lộ trình di chuyển</div>

                    <div class="search-wrapper">
                        <i class="fa-solid fa-circle-dot search-icon text-success"></i>
                        <input type="text" id="originInput" class="form-control search-input" placeholder="Nhập điểm đón..." autocomplete="off">
                        <button class="btn btn-light border gps-btn" onclick="useGPS('origin')"><i class="fa-solid fa-location-crosshairs text-primary"></i> GPS</button>
                        <div id="originDropdown" class="autocomplete-dropdown"></div>
                    </div>

                    <div class="search-wrapper">
                        <i class="fa-solid fa-location-dot search-icon text-danger"></i>
                        <input type="text" id="destInput" class="form-control search-input" placeholder="Nhập điểm đến..." autocomplete="off">
                        <button class="btn btn-light border gps-btn" onclick="useGPS('dest')"><i class="fa-solid fa-location-crosshairs text-primary"></i> GPS</button>
                        <div id="destDropdown" class="autocomplete-dropdown"></div>
                    </div>

                    <div class="section-title mt-4"><i class="fa-solid fa-car"></i> Lựa chọn Dịch vụ</div>
                    <select id="carTypeSelect" class="form-select mb-4 fw-bold" style="height: 50px;" onchange="requestPriceCalculation()">
                        <option value="1.0">🚘 Flash 4 Chỗ</option>
                        <option value="1.3">🚙 Flash 7 Chỗ</option>
                        <option value="1.5">🌟 Flash Premium</option>
                        <option value="2.0">👑 Flash VIP</option>
                    </select>

                    <div class="section-title"><i class="fa-solid fa-wallet"></i> Phương thức thanh toán</div>
                    <div class="payment-grid" id="paymentGrid">
                        <div class="payment-card active" data-val="cash" onclick="selectPayment('cash')">
                            <i class="fa-solid fa-money-bill-wave"></i><span>Tiền mặt</span>
                        </div>
                        <div class="payment-card" data-val="momo" onclick="selectPayment('momo')">
                            <i class="fa-solid fa-wallet"></i><span>MoMo</span>
                        </div>
                        <div class="payment-card" data-val="zalopay" onclick="selectPayment('zalopay')">
                            <i class="fa-solid fa-qrcode"></i><span>ZaloPay</span>
                        </div>
                        <div class="payment-card" data-val="card" onclick="selectPayment('card')">
                            <i class="fa-regular fa-credit-card"></i><span>Thẻ TD/GN</span>
                        </div>
                    </div>

                    <div class="section-title"><i class="fa-solid fa-ticket"></i> Mã Khuyến Mãi</div>
                    <div class="voucher-input-group">
                        <input type="text" id="voucherInput" class="form-control" placeholder="Mã (VD: DEMKHUYA)">
                        <button onclick="applyVoucher()">ÁP DỤNG</button>
                    </div>
                    <div id="voucherDesc" class="text-success"></div>

                    <div id="invoicePanel" class="invoice-box d-none mt-4">
                        <div class="d-flex justify-content-between align-items-center mb-4">
                            <h5 class="mb-0 fw-bold"><i class="fa-solid fa-receipt"></i> Phiếu Tính Tiền</h5>
                            <span class="badge bg-success">5D AI Validated</span>
                        </div>
                        <div class="inv-row"><span>Quãng đường:</span> <span class="inv-val" id="invDist">0 km</span></div>
                        <div class="inv-row"><span>Cước cơ bản (12k/km):</span> <span class="inv-val" id="invBase">0 đ</span></div>
                        <div class="inv-row">
                            <span title="Dựa trên Ngày, Giờ, Thời tiết, Giao thông, Nhu cầu">Hệ số Áp lực 5D: <i class="fa-solid fa-circle-question fa-sm"></i></span>
                            <span class="inv-val text-warning" id="invFuzzy">x1.00</span>
                        </div>
                        <div class="inv-row"><span>Hệ số Loại xe:</span> <span class="inv-val" id="invCar">x1.0</span></div>
                        <div class="inv-discount-row" id="invDiscountRow">
                            <span><i class="fa-solid fa-tags"></i> Khuyến mãi:</span>
                            <span id="invDiscountVal">-0 đ</span>
                        </div>
                        <div class="inv-total-row">
                            <span>TỔNG TIỀN:</span>
                            <span id="invTotal">0 đ</span>
                        </div>
                        <button class="btn btn-booking w-100 mt-4 shadow" onclick="findDriverAndStart()">XÁC NHẬN ĐẶT XE <i class="fa-solid fa-arrow-right ms-2"></i></button>
                    </div>

                    <div id="activeRidePanel" class="d-none mt-4">
                        <div class="d-flex justify-content-between align-items-center mb-3">
                            <h5 class="fw-bold text-primary mb-0"><i class="fa-solid fa-car-side"></i> Đang di chuyển</h5>
                            <span class="badge bg-success pulse-badge">Live Tracking</span>
                        </div>
                        <div class="p-3 bg-light mb-4 rounded d-flex align-items-center gap-3 border">
                            <img id="activeDriverAvatar" src="" alt="Driver" style="width: 60px; height: 60px; border-radius: 50%; border: 2px solid var(--accent); background: white;">
                            <div class="text-start flex-fill">
                                <h6 class="fw-bold mb-1" id="activeDriverName" style="font-size: 1.1rem;">Tên</h6>
                                <div class="small text-muted fw-bold">
                                    <i class="fa-solid fa-star text-warning"></i> <span id="activeDriverRating">5.0</span>
                                    <span class="mx-1">•</span>
                                    <span id="activeDriverPlate" class="text-dark bg-white border px-1 rounded">XX-XXX</span>
                                </div>
                            </div>
                        </div>

                        <!-- CỤM NÚT CHAT & GỌI ĐIỆN -->
                        <div class="action-btn-group">
                            <button class="action-btn btn-call" onclick="openCallUI()"><i class="fa-solid fa-phone-volume"></i> GỌI ĐIỆN</button>
                            <button class="action-btn btn-chat" onclick="openChatUI()"><i class="fa-solid fa-message"></i> NHẮN TIN</button>
                        </div>

                        <button class="btn btn-danger w-100 fw-bold py-3 fs-5 shadow-sm" onclick="endRide()"><i class="fa-solid fa-flag-checkered"></i> KẾT THÚC CHUYẾN</button>
                    </div>

                </div>
            </div>
        </div>
    </div>

    <!-- TÌM KIẾM TÀI XẾ MODAL -->
    <div id="driverModal">
        <div class="driver-card">
            <div id="radarState">
                <div class="radar-box">
                    <div class="radar-sweep"></div>
                    <i class="fa-solid fa-location-crosshairs"></i>
                </div>
                <h4 class="fw-bold mb-2">Đang quét tài xế...</h4>
                <p class="text-muted small">AI 5D đang tìm đối tác phù hợp nhất.</p>
                <div class="progress mt-3" style="height: 10px;">
                    <div class="progress-bar progress-bar-striped progress-bar-animated bg-primary" id="searchProgress" style="width: 0%"></div>
                </div>
            </div>
            <div id="foundState" style="display: none;" class="py-3">
                <div class="mb-3"><i class="fa-solid fa-circle-check text-success" style="font-size: 4rem;"></i></div>
                <h4 class="fw-bold mb-2">Chốt cuốc thành công!</h4>
                <p class="text-muted fw-bold" id="foundDriverNamePreview">Đang kết nối...</p>
            </div>
        </div>
    </div>

    <!-- ========================================== -->
    <!-- CÁC MODAL MỚI CHO CHAT, GỌI VÀ ĐÁNH GIÁ -->
    <!-- ========================================== -->

    <!-- 1. LIVE CHAT MODAL -->
    <div id="chatModal" class="overlay-modal">
        <div class="modal-content-box">
            <div class="chat-header">
                <div>
                    <h5 class="mb-0 fw-bold" id="chatTitleName">Tài xế</h5>
                    <small style="color: #cbd5e1;"><i class="fa-solid fa-circle text-success" style="font-size: 8px;"></i> Đang trực tuyến</small>
                </div>
                <button type="button" class="btn-close btn-close-white" onclick="closeChatUI()"></button>
            </div>
            <div class="chat-body" id="chatBody">
                <!-- Nội dung tin nhắn sẽ render ở đây -->
                <div class="msg-bubble msg-driver">Chào bạn, mình đã nhận chuyến. Bạn đợi mình chút xíu nhé!</div>
            </div>
            <div class="chat-footer">
                <input type="text" class="chat-input" id="chatInputMsg" placeholder="Nhập tin nhắn..." onkeypress="if(event.key === 'Enter') sendChatMessage()">
                <button class="chat-send" onclick="sendChatMessage()"><i class="fa-solid fa-paper-plane"></i></button>
            </div>
        </div>
    </div>

    <!-- 2. VOIP CALL MODAL -->
    <div id="callModal" class="overlay-modal">
        <div class="call-box">
            <h6 class="fw-bold text-success mb-2" style="letter-spacing: 2px;"><i class="fa-solid fa-shield-halved"></i> SECURE CALL</h6>
            <div class="call-avatar-wrapper ringing" id="callAvatarWrapper">
                <img id="callAvatarImg" src="" class="call-avatar" alt="Avatar">
            </div>
            <div>
                <h3 class="call-name" id="callTitleName">Tài xế</h3>
                <div class="call-status-text mt-2" id="callStatusText">Đang đổ chuông...</div>
            </div>
            <button class="call-btn-hangup" onclick="endCall()"><i class="fa-solid fa-phone-slash"></i></button>
        </div>
    </div>

    <!-- 3. RATING MODAL (ĐÁNH GIÁ 5 SAO) -->
    <div id="ratingModal" class="overlay-modal">
        <div class="modal-content-box">
            <div class="rating-box">
                <div style="font-size: 4rem; color: var(--success); margin-bottom: 10px;"><i class="fa-solid fa-circle-check"></i></div>
                <h3 class="rating-title">Chuyến đi hoàn tất!</h3>
                <p class="rating-subtitle">Vui lòng đánh giá trải nghiệm của bạn với <br><b id="ratingDriverName" style="color: var(--primary); font-size: 1.1rem;">Tài xế</b> <br>để giúp FlashGo cải thiện dịch vụ.</p>

                <div class="stars" id="starContainer">
                    <i class="fa-solid fa-star star-icon" data-val="5" onclick="setRating(5)"></i>
                    <i class="fa-solid fa-star star-icon" data-val="4" onclick="setRating(4)"></i>
                    <i class="fa-solid fa-star star-icon" data-val="3" onclick="setRating(3)"></i>
                    <i class="fa-solid fa-star star-icon" data-val="2" onclick="setRating(2)"></i>
                    <i class="fa-solid fa-star star-icon" data-val="1" onclick="setRating(1)"></i>
                </div>
                <input type="hidden" id="finalRatingValue" value="0">

                <textarea class="rating-comment" rows="3" placeholder="Chia sẻ thêm về chuyến đi của bạn (tùy chọn)..."></textarea>

                <button class="btn-submit-rating" onclick="submitRating()">GỬI ĐÁNH GIÁ <i class="fa-solid fa-check ms-2"></i></button>
                <!-- THÊM NÚT BỎ QUA ĐÁNH GIÁ -->
                <button class="btn btn-link w-100 mt-2 text-muted fw-bold text-decoration-none" onclick="skipRating()">Bỏ qua đánh giá</button>
            </div>
        </div>
    </div>

    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://unpkg.com/leaflet-routing-machine@latest/dist/leaflet-routing-machine.js"></script>

    <script>
        let map, routingControl;
        let markers = { origin: null, dest: null };
        let coords = { origin: null, dest: null };
        let currentRouteParams = { distanceKm: 0 };

        let aiData = {
            weather: 5, traffic: 50, hour: new Date().getHours(),
            dayOfWeek: new Date().getDay() === 0 ? 7 : new Date().getDay(),
            demand: 50
        };

        let searchTimeouts = { origin: null, dest: null };
        let currentVoucher = null;
        let currentBaseTotal = 0;
        let activeDriverMarkers = [];

        // --- CẬP NHẬT LOGIC KHỞI ĐỘNG VỚI SPLASH SCREEN ---
        window.addEventListener('load', () => {
            const statusBox = document.getElementById('splashStatus');
            setTimeout(() => { statusBox.innerText = "Đang kết nối OpenStreetMap..."; }, 800);
            setTimeout(() => { statusBox.innerText = "Đang tải dữ liệu Market Pressure..."; }, 1600);
            setTimeout(() => { statusBox.innerText = "Khởi tạo mạng nơ-ron mờ 5D..."; }, 2400);
            setTimeout(() => { statusBox.innerText = "Hoàn tất! Đang vào hệ thống..."; }, 3200);

            setTimeout(() => {
                const splash = document.getElementById('splashScreen');
                splash.style.opacity = '0';
                setTimeout(() => {
                    splash.style.visibility = 'hidden';
                    splash.style.display = 'none';

                    // Khởi tạo app sau khi ẩn màn hình chờ
                    initMap();
                    updateDashboard();
                    setInterval(updateDashboard, 60000);
                }, 800);
            }, 4000);
        });

        function showToast(message, type = 'success') {
            const container = document.getElementById('toastContainer');
            const toast = document.createElement('div');
            toast.className = `custom-toast ${type === 'success' ? 'toast-success' : 'toast-error'}`;
            toast.innerHTML = `<i class="fa-solid ${type === 'success' ? 'fa-circle-check' : 'fa-circle-xmark'} fs-4"></i> <div>${message}</div>`;
            container.appendChild(toast);
            setTimeout(() => toast.classList.add('show'), 10);
            setTimeout(() => { toast.classList.remove('show'); setTimeout(() => toast.remove(), 400); }, 3000);
        }

        function updateDashboard() {
            const now = new Date();
            aiData.hour = now.getHours() + now.getMinutes()/60;
            aiData.dayOfWeek = now.getDay() === 0 ? 7 : now.getDay();

            document.getElementById('dashTime').innerText = now.getHours().toString().padStart(2, '0') + ":" + now.getMinutes().toString().padStart(2, '0');
            const days = ["", "Thứ 2", "Thứ 3", "Thứ 4", "Thứ 5", "Thứ 6", "Thứ 7", "Chủ Nhật"];
            document.getElementById('dashDay').innerText = days[aiData.dayOfWeek];

            calculateMarketDemand();
        }

        function calculateMarketDemand() {
            let baseDemand = 40;
            if ((aiData.hour >= 7 && aiData.hour <= 9) || (aiData.hour >= 17 && aiData.hour <= 19)) baseDemand += 40;
            if (aiData.dayOfWeek >= 6 && aiData.hour >= 18) baseDemand += 30;
            if (aiData.weather >= 8) baseDemand += 25;

            baseDemand += Math.floor(Math.random() * 10) - 5;
            if(baseDemand > 100) baseDemand = 100;
            if(baseDemand < 10) baseDemand = 10;
            aiData.demand = baseDemand;

            let demandText = ""; let demandColor = "";
            if(baseDemand < 30) { demandText = "Thấp"; demandColor = "var(--success)"; }
            else if(baseDemand < 60) { demandText = "Bình thường"; demandColor = "var(--accent)"; }
            else if(baseDemand < 85) { demandText = "Cao"; demandColor = "var(--warning)"; }
            else { demandText = "Cực kỳ Quá Tải"; demandColor = "var(--danger)"; }

            const demandEl = document.getElementById('dashDemand');
            demandEl.innerText = `${demandText} (${baseDemand}%)`;
            demandEl.style.color = demandColor;
            aiData.traffic = Math.min(100, baseDemand + Math.floor(Math.random() * 20));
        }

        function initMap() {
            map = L.map('map', { zoomControl: false }).setView([10.776889, 106.700806], 14);
            L.control.zoom({ position: 'bottomright' }).addTo(map);
            L.tileLayer('https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png', { maxZoom: 20 }).addTo(map);

            routingControl = L.Routing.control({
                waypoints: [], routeWhileDragging: false, show: false,
                createMarker: function() { return null; },
                lineOptions: { styles: [{color: '#3b82f6', weight: 6, opacity: 0.9}] }
            }).addTo(map);

            routingControl.on('routesfound', function(e) {
                currentRouteParams.distanceKm = e.routes[0].summary.totalDistance / 1000;
                requestPriceCalculation();
            });

            map.on('click', function(e) {
                const {lat, lng} = e.latlng;
                L.popup({ closeButton: false })
                    .setLatLng(e.latlng)
                    .setContent(`
                        <div class="text-center p-2">
                            <b class="d-block mb-2">Gắn tọa độ này làm:</b>
                            <button class="btn btn-success btn-sm w-100 mb-2 fw-bold" onclick="selectLoc('origin', ${lat}, ${lng})">Điểm Đón</button>
                            <button class="btn btn-danger btn-sm w-100 fw-bold" onclick="selectLoc('dest', ${lat}, ${lng})">Điểm Đến</button>
                        </div>
                    `).openOn(map);
            });
        }

        async function getAddressFromCoords(lat, lng) {
            try {
                const res = await fetch(`https://nominatim.openstreetmap.org/reverse?lat=${lat}&lon=${lng}&format=jsonv2`);
                const data = await res.json();
                if (data && data.display_name) {
                    return data.display_name.split(', ').slice(0, 3).join(', ');
                }
            } catch (e) { console.error("Lỗi dịch tọa độ:", e); }
            return `Tọa độ: ${lat.toFixed(4)}, ${lng.toFixed(4)}`;
        }

        async function selectLoc(type, lat, lng) {
            map.closePopup();
            const inputEl = document.getElementById(`${type}Input`);
            inputEl.value = "Đang dịch tọa độ...";
            drawMarker(type, lat, lng);
            inputEl.value = await getAddressFromCoords(lat, lng);
        }

        function drawMarker(type, lat, lng) {
            const customIcon = L.divIcon({
                className: 'custom-marker',
                html: `<div class="${type}-marker"><div class="marker-pin"></div></div>`,
                iconSize: [30, 42], iconAnchor: [15, 42]
            });
            if (markers[type]) map.removeLayer(markers[type]);
            markers[type] = L.marker([lat, lng], {icon: customIcon}).addTo(map);
            coords[type] = L.latLng(lat, lng);

            if(type === 'origin') fetchRealWeather(lat, lng);

            if (coords.origin && coords.dest) {
                map.fitBounds(L.latLngBounds([coords.origin, coords.dest]), { padding: [50, 50] });
                routingControl.setWaypoints([coords.origin, coords.dest]);
            }
        }

        document.getElementById('originInput').addEventListener('input', (e) => handleSearch(e.target.value, 'origin'));
        document.getElementById('destInput').addEventListener('input', (e) => handleSearch(e.target.value, 'dest'));

        function handleSearch(query, type) {
            const dd = document.getElementById(`${type}Dropdown`);
            if (query.length < 3) { dd.style.display = 'none'; return; }
            clearTimeout(searchTimeouts[type]);
            searchTimeouts[type] = setTimeout(async () => {
                try {
                    const res = await fetch(`https://nominatim.openstreetmap.org/search?format=json&limit=5&q=${encodeURIComponent(query)}&countrycodes=vn`);
                    const results = await res.json();
                    dd.innerHTML = '';
                    if (results.length > 0) {
                        results.forEach(item => {
                            const div = document.createElement('div');
                            div.className = 'autocomplete-item';
                            div.innerHTML = `<i class="fa-solid fa-location-dot"></i><div class="autocomplete-text">${item.display_name}</div>`;
                            div.onclick = () => {
                                dd.style.display = 'none';
                                document.getElementById(`${type}Input`).value = item.display_name.split(', ').slice(0, 3).join(', ');
                                drawMarker(type, parseFloat(item.lat), parseFloat(item.lon));
                            };
                            dd.appendChild(div);
                        });
                        dd.style.display = 'block';
                    } else { dd.style.display = 'none'; }
                } catch (e) {}
            }, 600);
        }

        function useGPS(type) {
            if (!navigator.geolocation) return showToast("Trình duyệt không hỗ trợ GPS!", "error");

            const inputEl = document.getElementById(`${type}Input`);
            inputEl.value = "Đang lấy GPS...";

            navigator.geolocation.getCurrentPosition(
                async (pos) => {
                    const lat = pos.coords.latitude;
                    const lng = pos.coords.longitude;
                    drawMarker(type, lat, lng);

                    inputEl.value = "Đang dịch tọa độ...";
                    inputEl.value = await getAddressFromCoords(lat, lng);

                    showToast("Lấy vị trí GPS thành công!");
                },
                () => { showToast("Vui lòng cấp quyền vị trí!", "error"); inputEl.value = ""; }
            );
        }

        async function fetchRealWeather(lat, lng) {
            try {
                const res = await fetch(`https://api.open-meteo.com/v1/forecast?latitude=${lat}&longitude=${lng}&current_weather=true`);
                const data = await res.json();
                const code = data.current_weather.weathercode;

                if (code <= 1) aiData.weather = 1;
                else if (code <= 48) aiData.weather = 5;
                else if (code <= 67) aiData.weather = 8;
                else aiData.weather = 10;

                document.getElementById('dashWeather').innerText = data.current_weather.temperature + "°C";
                calculateMarketDemand();
                if(currentRouteParams.distanceKm > 0) requestPriceCalculation();
            } catch(e) {}
        }

        function selectPayment(method) {
            document.querySelectorAll('.payment-card').forEach(c => c.classList.remove('active'));
            document.querySelector(`.payment-card[data-val="${method}"]`).classList.add('active');
        }

        function applyVoucher() {
            const code = document.getElementById('voucherInput').value.trim().toUpperCase();
            if(!code) return showToast("Nhập mã KM", "error");
            fetch('/validate_voucher', {
                method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ code: code })
            }).then(res => res.json()).then(data => {
                if(data.valid) {
                    currentVoucher = data.voucher;
                    document.getElementById('voucherDesc').innerHTML = `<i class="fa-solid fa-check"></i> ${data.voucher.desc}`;
                    showToast("Áp dụng mã thành công!");
                    renderFinalPrice();
                } else {
                    currentVoucher = null;
                    document.getElementById('voucherDesc').innerText = "";
                    showToast("Mã không hợp lệ", "error");
                    renderFinalPrice();
                }
            });
        }

        function requestPriceCalculation() {
            if(currentRouteParams.distanceKm === 0) return;
            const carMult = parseFloat(document.getElementById('carTypeSelect').value);

            fetch('/calculate_5d', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({
                    distance: currentRouteParams.distanceKm,
                    weather: aiData.weather,
                    traffic: aiData.traffic,
                    hour: aiData.hour,
                    day: aiData.dayOfWeek,
                    demand: aiData.demand,
                    car_multiplier: carMult
                })
            })
            .then(res => res.json())
            .then(data => {
                document.getElementById('invoicePanel').classList.remove('d-none');
                document.getElementById('invDist').innerText = `${data.distance.toFixed(2)} km`;
                document.getElementById('invBase').innerText = `${data.base_fare.toLocaleString('vi-VN')} đ`;
                document.getElementById('invFuzzy').innerText = `x${data.fuzzy_multiplier.toFixed(2)}`;
                document.getElementById('invCar').innerText = `x${carMult.toFixed(1)}`;
                currentBaseTotal = Number(data.final_price);
                renderFinalPrice();
            });
        }

        function renderFinalPrice() {
            let final = currentBaseTotal || 0;
            let discountAmount = 0;
            const discRow = document.getElementById('invDiscountRow');

            if(currentVoucher) {
                if(currentVoucher.type === 'percent') discountAmount = final * currentVoucher.value;
                else if(currentVoucher.type === 'fixed') discountAmount = currentVoucher.value;

                if(discountAmount > currentVoucher.max_discount) discountAmount = currentVoucher.max_discount;
                final -= discountAmount;
                if(final < 0) final = 0;

                discRow.style.display = 'flex';
                document.getElementById('invDiscountVal').innerText = `-${Math.round(discountAmount).toLocaleString('vi-VN')} đ`;
            } else { discRow.style.display = 'none'; }

            final = Math.round(final / 1000) * 1000;
            document.getElementById('invTotal').innerText = `${final.toLocaleString('vi-VN')} đ`;
        }

        function findDriverAndStart() {
            if(!coords.origin || !coords.dest) return showToast("Chọn điểm đón và đến!", "error");

            document.getElementById('driverModal').style.display = 'flex';
            document.getElementById('radarState').style.display = 'block';
            document.getElementById('foundState').style.display = 'none';

            let progress = 0;
            const bar = document.getElementById('searchProgress');

            const interval = setInterval(() => {
                progress += Math.random() * 25;
                if(progress > 100) progress = 100;
                bar.style.width = `${progress}%`;

                if(progress === 100) {
                    clearInterval(interval);
                    fetch('/get_random_driver')
                        .then(res => res.json())
                        .then(driver => {
                            document.getElementById('radarState').style.display = 'none';
                            document.getElementById('foundState').style.display = 'block';
                            document.getElementById('foundDriverNamePreview').innerText = `Tài xế ${driver.name} đã nhận chuyến!`;
                            showToast("Đã ghép xe thành công!");

                            setTimeout(() => {
                                document.getElementById('driverModal').style.display = 'none';
                                document.getElementById('activeDriverName').innerText = driver.name;
                                document.getElementById('activeDriverRating').innerText = driver.rating;
                                document.getElementById('activeDriverPlate').innerText = driver.plate;
                                document.getElementById('activeDriverAvatar').src = `https://api.dicebear.com/7.x/avataaars/svg?seed=${driver.name}&backgroundColor=b6e3f4`;

                                document.getElementById('invoicePanel').classList.add('d-none');
                                document.getElementById('activeRidePanel').classList.remove('d-none');
                            }, 2000);
                        });
                }
            }, 300);
        }

        /* ========================================================= */
        /* --- JS LOGIC MỚI CHO CHAT, GỌI ĐIỆN VÀ RATING SAU CHUYẾN --- */
        /* ========================================================= */

        // 1. LIVE CHAT SYSTEM
        function openChatUI() {
            document.getElementById('chatTitleName').innerText = document.getElementById('activeDriverName').innerText;
            document.getElementById('chatModal').style.display = 'flex';
        }

        function closeChatUI() {
            document.getElementById('chatModal').style.display = 'none';
        }

        function sendChatMessage() {
            const inputEl = document.getElementById('chatInputMsg');
            const text = inputEl.value.trim();
            if(!text) return;

            const chatBody = document.getElementById('chatBody');
            chatBody.innerHTML += `<div class="msg-bubble msg-user">${text}</div>`;
            inputEl.value = "";
            chatBody.scrollTop = chatBody.scrollHeight;

            // Auto-reply giả lập từ AI
            setTimeout(() => {
                if(document.getElementById('chatModal').style.display === 'flex') {
                    const replies = ["Dạ vâng ạ!", "Mình sắp tới rồi nhé.", "Ok bạn, đợi mình xíu nha.", "Mình thấy bạn rồi, chuẩn bị ra nha."];
                    const reply = replies[Math.floor(Math.random() * replies.length)];
                    chatBody.innerHTML += `<div class="msg-bubble msg-driver">${reply}</div>`;
                    chatBody.scrollTop = chatBody.scrollHeight;
                }
            }, 1500 + Math.random() * 1000);
        }

        // 2. VOIP CALL SYSTEM
        let callTimerInterval;
        let callSeconds = 0;

        function openCallUI() {
            document.getElementById('callTitleName').innerText = document.getElementById('activeDriverName').innerText;
            document.getElementById('callAvatarImg').src = document.getElementById('activeDriverAvatar').src;

            document.getElementById('callAvatarWrapper').classList.add('ringing');
            document.getElementById('callStatusText').innerText = 'Đang đổ chuông...';
            document.getElementById('callModal').style.display = 'flex';

            // Giả lập tài xế bắt máy sau 3 giây
            setTimeout(() => {
                if(document.getElementById('callModal').style.display === 'flex') {
                    document.getElementById('callAvatarWrapper').classList.remove('ringing');
                    startCallTimer();
                }
            }, 3000);
        }

        function startCallTimer() {
            callSeconds = 0;
            document.getElementById('callStatusText').innerText = '00:00';
            callTimerInterval = setInterval(() => {
                callSeconds++;
                const m = String(Math.floor(callSeconds / 60)).padStart(2, '0');
                const s = String(callSeconds % 60).padStart(2, '0');
                document.getElementById('callStatusText').innerText = `${m}:${s}`;
            }, 1000);
        }

        function endCall() {
            clearInterval(callTimerInterval);
            document.getElementById('callStatusText').innerText = 'Cuộc gọi đã kết thúc';
            setTimeout(() => {
                document.getElementById('callModal').style.display = 'none';
            }, 1000);
        }

        // 3. POST-RIDE RATING SYSTEM (OPTIONAL)
        function endRide() {
            document.getElementById('activeRidePanel').classList.add('d-none');

            // Hiện Modal đánh giá
            document.getElementById('ratingDriverName').innerText = document.getElementById('activeDriverName').innerText;
            document.getElementById('ratingModal').style.display = 'flex';

            // Reset trạng thái sao
            document.querySelectorAll('.star-icon').forEach(s => s.classList.remove('active'));
            document.getElementById('finalRatingValue').value = 0;
            document.querySelector('.rating-comment').value = "";
        }

        function setRating(val) {
            document.getElementById('finalRatingValue').value = val;
            const stars = document.querySelectorAll('.star-icon');

            // Highlight các ngôi sao từ 1 đến giá trị đã chọn
            stars.forEach(s => {
                if(parseInt(s.getAttribute('data-val')) <= val) {
                    s.classList.add('active');
                    s.style.color = 'var(--warning)';
                } else {
                    s.classList.remove('active');
                    s.style.color = '#cbd5e1';
                }
            });
        }

        // HÀM MỚI ĐỂ BỎ QUA ĐÁNH GIÁ
        function skipRating() {
            document.getElementById('finalRatingValue').value = 0;
            submitRating();
        }

        function submitRating() {
            const ratingValue = document.getElementById('finalRatingValue').value;

            // ĐÃ GỠ BỎ ĐIỀU KIỆN ÉP CHỌN SAO Ở ĐÂY
            document.getElementById('ratingModal').style.display = 'none';

            if (ratingValue > 0) {
                showToast("Đã gửi đánh giá " + ratingValue + " sao. Cảm ơn phản hồi của bạn!", "success");
            } else {
                showToast("Đã hoàn tất chuyến đi. Hẹn gặp lại bạn!", "success");
            }

            // Thực hiện quy trình Reset Hệ thống (như endRide cũ)
            setTimeout(() => {
                document.getElementById('originInput').value = "";
                document.getElementById('destInput').value = "";
                if(markers.origin) map.removeLayer(markers.origin);
                if(markers.dest) map.removeLayer(markers.dest);
                routingControl.setWaypoints([]);
                coords = {origin: null, dest: null};
                currentRouteParams.distanceKm = 0;
                document.getElementById('voucherInput').value = "";
                currentVoucher = null;
            }, 600);
        }

    </script>
</body>
</html>
"""

# ==============================================================================
# PHẦN 4: FLASK SERVER API ROUTES CHO 5D AI
# -> GIỮ NGUYÊN 100% CẤU TRÚC ROUTER
# ==============================================================================
@app.route('/')
def home():
    return render_template_string(HTML_CODE)

@app.route('/calculate_5d', methods=['POST'])
def calculate_5d():
    try:
        data = request.json
        dist_km = float(data.get('distance', 0))
        weather = float(data.get('weather', 5))
        traffic = float(data.get('traffic', 50))
        hour = float(data.get('hour', 12))
        day = float(data.get('day', 1))
        demand = float(data.get('demand', 50))
        car_mult = float(data.get('car_multiplier', 1.0))

        fuzzy_mult = calculate_5d_fuzzy(weather, traffic, hour, day, demand)
        base_price = 15000 + (dist_km * 12000)
        final_price = base_price * fuzzy_mult * car_mult

        return jsonify({
            'distance': dist_km,
            'base_fare': round(base_price / 1000) * 1000,
            'fuzzy_multiplier': fuzzy_mult,
            'final_price': round(final_price / 1000) * 1000
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/validate_voucher', methods=['POST'])
def validate_voucher():
    try:
        code = request.json.get('code', '').upper()
        if code in VOUCHERS_DB:
            return jsonify({'valid': True, 'voucher': VOUCHERS_DB[code]})
        return jsonify({'valid': False})
    except:
        return jsonify({'valid': False}), 500

@app.route('/get_random_driver', methods=['GET'])
def get_random_driver():
    return jsonify(random.choice(DRIVERS_DB))

# ==============================================================================
# KHỞI CHẠY LOCALTUNNEL VÀ SERVER
# ==============================================================================
PORT = 5000

def run_server():
    server = make_server('0.0.0.0', PORT, app)
    server.serve_forever()

if __name__ == '__main__':
    threading.Thread(target=run_server, daemon=True).start()
    time.sleep(2)

    try:
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        print("\n" + "="*80)
        print("🚀 FLASHGO ENTERPRISE 5D AI ĐÃ KHỞI ĐỘNG 🚀")
        print("="*80)
        print(f"🔑 MẬT KHẨU CỦA BẠN CHO LOCALTUNNEL LÀ: {ip}")
        print("🌐 CLICK VÀO LINK BÊN DƯỚI ĐỂ TRẢI NGHIỆM")
        print("="*80 + "\n")

        !npx localtunnel --port {PORT}
    except Exception as e:
        print(f"❌ Lỗi thiết lập mạng: {e}")